*Notebook 10 of 12 in `notebooks/scripts_text/`. Parameters: `analysis/scripts_text/config.yaml`; decisions and run notes: `analysis/scripts_text/LOG.md`. The static page `site/scripts/index.html` is built from the same outputs.*

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parents[1]   # notebooks/scripts_text -> repo root
sys.path.insert(0, str(REPO))

import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display

from analysis.scripts_text import s10_franchise as step

pio.renderers.default = "notebook_connected"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## Step 10 · Franchise and leakage checks

**What we do.** We flag likely franchise films two ways: Wikidata's "follows" property (the film is a sequel;
clean, but it misses first films) and membership of a named media franchise (catches first films too). The
"series" property is reported but not used, because it also holds lists such as "100 greatest films". We
rerun the primary comparison without franchise films, without sequels only, and without scripts that might be
post-release transcripts (almost no action description, or fewer than 20 scenes), each with its own
200-shuffle null.

**Why.** Sequels reuse characters, places and catchphrases, and franchise films are often hits. If a few
franchises drive the "successful" vocabulary, the result is about franchises, not scripts. Name removal works
script by script, so a sequel's script can still contain a character name that is only a speaker in the
first film. A transcript written from the finished film is not the script that was green-lit, which matters
for anything we say about scripts predicting success.

**What it measures.** Whether the top terms and the overall count test survive without these films.

**What it misses.** Both franchise flags are incomplete, and the transcript rule is a heuristic.

In [ ]:
res = step.run()   # recomputes this step and saves its outputs

**Figure.** z of the primary count test (terms at least twice as frequent in one group) against its own 200-shuffle stratified null, for the full comparison and with each set of films removed. The box (exploratory control) drops the same number of random non-franchise films 50 times. Inside the grey band the real groups are no more different than random ones.

In [ ]:
step.figures(res)["z"].show()

**Figure.** Overlap of the top-50 lists with the primary run after dropping each set of films. Bars near 1: the result doesn't depend on those films.

In [ ]:
step.figures(res)["overlap"].show()

**Table.** Each rerun: films dropped, group sizes, overlap with the primary run, and the count test against its own stratified null.

In [ ]:
display(step.tables(res)["results"])

,run,dropped,successful,unsuccessful,jaccard_high_vs_primary,jaccard_low_vs_primary,terms_above_threshold,null_mean,null_std,z,p_empirical,top10_high
0,primary,0,290,236,1.000,1.000,210,183.390,12.964,2.053,0.040,"je, pas, rocky, c'est, balboa, il, parker, yah, chancellor, vous"
1,without likely franchise films,93,213,220,0.493,0.471,221,213.935,14.025,0.504,0.299,"je, pas, ce, c'est, il, yah, vous, tu, stark, mais"
2,without sequels only,63,239,224,0.538,0.639,207,208.470,12.266,-0.120,0.577,"je, pas, c'est, ce, il, yah, tu, vous, stark, parker"
3,without possible transcripts,24,278,224,0.754,0.667,204,184.635,12.797,1.513,0.095,"je, pas, rocky, balboa, c'est, chancellor, parker, il, yah, vous"


**Plain-language reading** (generated from the saved numbers):

In [ ]:
display(Markdown(step.reading(res)))

Of the 526 films in the primary comparison, 93 are likely franchise films (63 follow another film on Wikidata, 53 belong to a named media franchise); they make up 27% of the successful and 7% of the unsuccessful group. Dropping them, the top-50 lists overlap the primary ones with Jaccard 0.49 (successful) and 0.47 (unsuccessful), and the count of terms above 2× has z = 0.50 against 200 stratified shuffles (p = 0.299). As an exploratory control, dropping the same number of films (77 successful, 16 unsuccessful) at random among the non-franchise films, 50 times, gives a median z of 2.94 (90% between 1.75 and 4.36); 0% of random drops fall as low as dropping the franchise films does, so losing those particular films, not just losing films, is what removes the excess. 24 scripts look like they could be transcripts (less than 15% action words, or fewer than 20 scenes); without them the overlap is 0.75 and z = 1.51. Without franchise films, Step 9's classifier gives AUC 0.576 for text against 0.662 for the baseline (3 repeats). The Wikidata series property is reported but not used: it marks 142 of these films and includes lists, not just series.